<a href="https://colab.research.google.com/github/pedadarohan-dot/Multi_Agent_CTR/blob/main/Multi_agent_CTR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import io
import sys
import traceback
import contextlib
from typing import TypedDict, Literal, Any
import pandas as pd
import gradio as gr
from dotenv import load_dotenv

# 1. Update core packages with compatible versions - added 'langchain'
!pip install -U -q "langchain" "langchain-google-genai" "langchain-core" "langgraph>=0.1.5,<0.2.0"

# IMPORTANT: If the import below fails with 'ImportError' or 'ModuleNotFoundError',
# you MUST go to Runtime -> Restart session and then run this cell again.

from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import MemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate

print("Successfully Completed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.2/164.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.3/718.3 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 31.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires n

In [ ]:
load_dotenv()
import google.generativeai as genai
from google.colab import userdata

# Securely fetch your API key from Colab secrets
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
except Exception as e:
    print("Warning: GOOGLE_API_KEY not found in secrets. Please add it to use Gemini.")

# Initialize the LangChain Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

print("Gemini Model Initialized")

Gemini Model Initialized


In [ ]:
class GraphState(TypedDict):
  task: str
  uploaded_file: Any
  dataset_info: str
  instructions: str
  code: str
  exec_output: str
  exec_error: str
  attempts: int
  suggestions: str

In [ ]:
def ui_input_agent(state: GraphState) -> GraphState:
  global df
  df = load_file(state.get("uploaded_file"))
  if isinstance(df, pd.DataFrame) and not df.empty:
    cols = ", ".join(df.columns.tolist())
    dataset_info = f"The uploaded file contains the following columns: {cols}."
  else:
    dataset_info = "No data was uploaded (empty Dataframe)."
  return {
      "task": state["task"],
      "uploaded_file": state.get("uploaded_file"),
      "dataset_info": dataset_info,
      "attempts": 0,
      "suggestions": "",
  }


In [ ]:
planner_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "... system instructions ..."),
        ("human", "Task description: {task}\nDataset info: {dataset_info}\n\nProvide the numbered instructions."),
    ]
)

def planner_agent(state: GraphState) -> GraphState:
    prompt = planner_prompt.format_messages(
        task=state["task"],
        dataset_info=state["dataset_info"],
    )
    response = llm.invoke(prompt)
    instructions = response.content.strip()
    return {"instructions": instructions}

In [ ]:
coder_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "... system instructions ..."),
        ("human", "Instructions:\n{instructions}\n\nWrite the code now:"),
    ]
)

def coder_agent(state: GraphState) -> GraphState:
    prompt = coder_prompt.format_messages(instructions=state["instructions"])
    response = llm.invoke(prompt)
    code = response.content.strip()
    if code.startswith("```python"):
        code = code[len("```python"):].strip()
    if code.startswith("```"):
        code = code[3:].strip()
    if code.endswith("```"):
        code = code[:-3].strip()
    return {"code": code}

In [ ]:
def executor_agent(state: GraphState) -> GraphState:
    code = state.get("code", "")
    if not code:
        return {"exec_output": "", "exec_error": "No code was provided by the coder."}
    exec_namespace = dict(globals())
    exec_namespace.update({"__name__": "__main__"})
    stdout_buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_buf):
            exec(code, exec_namespace)
        return {"exec_output": stdout_buf.getvalue(), "exec_error": ""}
    except Exception:
        tb = traceback.format_exc()
        return {"exec_output": "", "exec_error": tb}

In [ ]:
reviewer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "... system instructions ..."),
        ("human", "Error:\n```\n{error}\n```\n\nOriginal code:\n```\n{code}\n```"),
    ]
)

def reviewer_agent(state: GraphState) -> GraphState:
    prompt = reviewer_prompt.format_messages(error=state["exec_error"], code=state["code"])
    response = llm.invoke(prompt)
    reply = response.content.strip()
    if reply.upper().startswith("SUGGEST:"):
        suggestion = reply[len("SUGGEST:"):].strip()
        return {"suggestions": suggestion, "code": state["code"]}
    corrected = reply
    if corrected.startswith("```python"):
        corrected = corrected[len("```python"):].strip()
    if corrected.startswith("```"):
        corrected = corrected[3:].strip()
    if corrected.endswith("```"):
        corrected = corrected[:-3].strip()
    return {"code": corrected, "suggestions": ""}

In [ ]:
# Workflow definition
MAX_ATTEMPTS = 5
workflow = StateGraph(GraphState)

workflow.add_node("input", ui_input_agent)
workflow.add_node("planner", planner_agent)
workflow.add_node("coder", coder_agent)
workflow.add_node("executor", executor_agent)
workflow.add_node("reviewer", reviewer_agent)

workflow.add_edge(START, "input")
workflow.add_edge("input", "planner")
workflow.add_edge("planner", "coder")
workflow.add_edge("coder", "executor")

def after_executor(state: GraphState) -> Literal["reviewer", END]:
    if state["exec_error"] and state["attempts"] < MAX_ATTEMPTS:
        return "reviewer"
    return END

workflow.add_conditional_edges("executor", after_executor, {"reviewer": "reviewer", END: END})
workflow.add_edge("reviewer", "executor")
app = workflow.compile()

In [ ]:
def _normalise_path(file_obj) -> str:
    if file_obj is None:
        return ""
    if isinstance(file_obj, str):
        return file_obj
    return getattr(file_obj, "name", "")


def load_file(file_obj):
    path = _normalise_path(file_obj)
    if not path:
        return pd.DataFrame()
    ext = path.lower().split(".")[-1]
    try:
        if ext == "csv":
            df = pd.read_csv(path)
        elif ext in ("xlsx", "xls"):
            df = pd.read_excel(path)
        else:
            raise ValueError("Unsupported file type. Please upload CSV or Excel.")
    except Exception as exc:
        raise ValueError(f"Failed to read the uploaded file: {exc}") from exc
    return df.drop(columns="Date", errors="ignore")

def run_workflow(task: str, uploaded_file) -> dict:
    final_state = app.invoke({"task": task, "uploaded_file": uploaded_file})
    result = {
        "planner": final_state.get("instructions", ""),
        "coder": final_state.get("code", ""),
        "executor_output": final_state.get("exec_output", ""),
        "executor_error": final_state.get("exec_error", ""),
        "reviewer": final_state.get("suggestions", ""),
        "final_output": "",
    }
    if final_state.get("exec_error"):
        if final_state.get("suggestions"):
            result["final_output"] = (
                "❗️  The system could not automatically fix the code.\n"
                f"💡  Suggested next step for the human:\n{final_state['suggestions']}"
            )
        else:
            result["final_output"] = (
                "❗️  Execution failed after the maximum number of attempts.\n"
                f"Error was:\n{final_state['exec_error']}"
            )
    else:
        result["final_output"] = "✅  Code ran successfully!  Output:\n" + final_state["exec_output"]
    return result

In [ ]:
with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("""
        # 🤖 Data‑Science Assistant (LangGraph + Watsonx)
        1️⃣ Upload a CSV or Excel file.
        2️⃣ Describe the analysis / model you want in plain English.
        3️⃣ Press Run – the assistant will plan, code, execute, review and finally give you the result.
    """)

    with gr.Row():
        file_input = gr.File(label="📂 Upload CSV / Excel (optional)", file_types=[".csv", ".xlsx", ".xls"])
        task_input = gr.Textbox(label="📝 Task description", placeholder="e.g. train a linear regression model", lines=3)

    run_btn = gr.Button("🚀 Run", variant="primary")

    planner_box = gr.Textbox(label="🗒️ Planner – numbered steps", lines=6)
    coder_box   = gr.Code(label="👩‍💻 Coder – generated Python code", language="python", lines=12)
    executor_out = gr.Textbox(label="⚙️ Executor – stdout", lines=6)
    executor_err = gr.Textbox(label="❌ Executor – error (if any)", lines=6)
    reviewer_box = gr.Textbox(label="🧐 Reviewer – suggestion (if any)", lines=6)
    final_box    = gr.Textbox(label="🎉 Final output", lines=8)

    def on_click(task, uploaded_file):
        out = run_workflow(task, uploaded_file)
        return (
            out["planner"], out["coder"], out["executor_output"],
            out["executor_error"], out["reviewer"], out["final_output"],
        )

    run_btn.click(fn=on_click, inputs=[task_input, file_input], outputs=[planner_box, coder_box, executor_out, executor_err, reviewer_box, final_box])

/tmp/ipykernel_532/984459497.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default()) as demo:
